In [2]:
import os
from utils_ipynb import get_local_folder
notebook_dir = get_local_folder()
os.chdir(f"{notebook_dir}/..")

os.environ["CUDA_VISIBLE_DEVICES"] = "5"


## Load Multi-Krum and Get True Label

In [3]:
import ast


ckpt_name_dir_map = {
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_multi-krum": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-20_23-57-25",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_multi-krum": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-29_02-15-17",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_trimmed_mean": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-31_00-48-08",
}


# ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_multi-krum"
ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_multi-krum"
ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_trimmed_mean"
import re
#Load True Label

exp_log = open(f"{ckpt_name_dir_map[ckpt_name]}/main_kma.log", "r").readlines()

cur_round = -1
match_result = {}
for line in exp_log:
    if "Round" in line:
        # match "Round 1 : [0, 2, 4, 6, 9]"
        round_str = re.search(r'Round (\d+)', line)

        pattern = r"Round\s+(?P<round_number>\d+)\s*:\s*(?P<list>\[[\d,\s]+\])"
        match = re.search(pattern, line)

        if match:
            round_number = match.group("round_number")
            list_string = match.group("list")
            list_data = ast.literal_eval(list_string)
            # print(f"Round Number: {round_number}, List: {list_string}")

            cur_round = round_number
            match_result[cur_round + "_participants"] = list_data
    
    if "Multi-Krum" in line:
        # match "Multi-Krum: [0, 2, 4, 6, 9]"
        pattern = r"Multi-Krum select\s*(\[[\d,\s]+\])"
        match = re.search(pattern, line)
        if match:
            list_string = match.group(1)
            list_data = ast.literal_eval(list_string)
            # print(f"List: {list_string}")

            match_result[cur_round + "_selected"] = list_data

print(match_result)

{'1_participants': [0, 2, 4, 6, 9], '1_selected': [0, 2, 9, 6], '2_participants': [0, 1, 2, 3, 4], '2_selected': [3, 4, 1], '3_participants': [0, 1, 2, 7, 8], '3_selected': [7, 2, 0], '4_participants': [2, 3, 4, 7, 8], '4_selected': [2, 3, 4, 7, 8], '5_participants': [1, 3, 4, 5, 9], '5_selected': [9, 3, 4, 5], '6_participants': [4, 5, 6, 7, 9], '6_selected': [4, 5, 6, 7, 9], '7_participants': [1, 2, 6, 7, 9], '7_selected': [1, 2, 9, 7], '8_participants': [0, 2, 5, 6, 9], '8_selected': [0, 2, 5, 9], '9_participants': [1, 3, 5, 6, 7], '9_selected': [7, 3, 5, 6], '10_participants': [1, 4, 5, 6, 7], '10_selected': [1, 4, 5, 6], '11_participants': [0, 3, 4, 6, 9], '11_selected': [0, 3, 4, 9], '12_participants': [3, 4, 7, 8, 9], '12_selected': [3, 4, 7, 8, 9], '13_participants': [1, 3, 4, 5, 7], '13_selected': [1, 3, 7, 5], '14_participants': [1, 2, 4, 5, 9], '14_selected': [1, 2, 9, 5], '15_participants': [1, 2, 3, 5, 8], '15_selected': [1, 2, 3, 5], '16_participants': [0, 1, 3, 6, 8], '16

In [ ]:
import json
import torch 
total_round = 20
max_compromised_clients = 2
number_of_clients = 5
threshold_factor = 0.5
device = "cpu"
metric_name = "total_acc"
deviation = 0.05

ckpt_dir = ckpt_name_dir_map[ckpt_name]
all_metrics_path = f"{ckpt_dir}/evaluation_false_acc_NoSysQA.json"
all_metrics = json.load(open(all_metrics_path, "r"))
# parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']
parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']

fact = list(all_metrics[0]["global"].keys())[0]


def get_params(keys, param_dict):
    return param_dict[keys[0]]

t = 0

client_results = {}

for round in range(1, total_round+1):
    participants_clients = match_result[f"{round}_participants"]
    selected_clients = match_result[f"{round}_selected"]

    filtered_clients = [client for client in participants_clients if client not in selected_clients]

    attack_client_idxs = []
    for i in range(max_compromised_clients):
        if i in participants_clients:
            attack_client_idxs.append(i)

    match_result[f"{round}_filtered"] = filtered_clients
    match_result[f"{round}_attack_client_idxs"] = attack_client_idxs
    if len(attack_client_idxs) > 0:
        select_attack_idx = attack_client_idxs[0]
        
        t_nd = round - 1

        #上一次攻击时，初始全局模型
        theta_t = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[-2]

        #当前轮次初始全局模型
        theta_t_nd = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t_nd+1}.pth"), map_location=torch.device(device))[-2]

        #上一轮次恶意客户端模型
        theta_t_local = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[select_attack_idx]

        params_t = get_params(parameter_keys, theta_t)
        params_t_nd = get_params(parameter_keys, theta_t_nd)
        params_t_local = get_params(parameter_keys, theta_t_local)
        
        
        from utils.utils import read_indicator

        performance_feedback, param_feedback = read_indicator(t, t_nd, params_t, params_t_nd, params_t_local, number_of_clients=number_of_clients, threshold_factor=threshold_factor, all_metrics=all_metrics, fact=fact, metric_name=metric_name, deviation=deviation)

        t_participants = match_result[f"{t+1}_participants"]
        t_selected = match_result[f"{t+1}_selected"]
        t_filtered = match_result[f"{t+1}_filtered"]
        t_attack_client_idxs = match_result[f"{t+1}_attack_client_idxs"]

        print(f"Round {t+1} : participants: {t_participants}, selected: {t_selected}, filtered: {t_filtered}, attack_client_idxs: {t_attack_client_idxs}, performance_feedback: {performance_feedback}, param_feedback: {param_feedback}")

        t = t_nd
        # break

    




Round 1 : participants: [0, 2, 4, 6, 9], selected: [0, 2, 9, 6], filtered: [4], attack_client_idxs: [0], feedback: False
Round 1 : participants: [0, 2, 4, 6, 9], selected: [0, 2, 9, 6], filtered: [4], attack_client_idxs: [0], feedback: False
metric_t_nd: 0.0, metric_t: 0.0
p: 0.25075915455818176, threshold: 0.1
Round 2 : participants: [0, 1, 2, 3, 4], selected: [3, 4, 1], filtered: [0, 2], attack_client_idxs: [0, 1], feedback: True
metric_t_nd: 0.42, metric_t: 0.0
Round 3 : participants: [0, 1, 2, 7, 8], selected: [7, 2, 0], filtered: [1, 8], attack_client_idxs: [0, 1], feedback: True
metric_t_nd: 0.38, metric_t: 0.42
p: 0.042838871479034424, threshold: 0.1
Round 5 : participants: [1, 3, 4, 5, 9], selected: [9, 3, 4, 5], filtered: [1], attack_client_idxs: [1], feedback: False
metric_t_nd: 0.86, metric_t: 0.38
Round 7 : participants: [1, 2, 6, 7, 9], selected: [1, 2, 9, 7], filtered: [6], attack_client_idxs: [1], feedback: True
metric_t_nd: 0.96, metric_t: 0.86
Round 8 : participants: [

## Trimmed Mean

In [4]:
import ast


ckpt_name_dir_map = {
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_multi-krum": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-20_23-57-25",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_multi-krum": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-29_02-15-17",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_trimmed_mean": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-31_00-48-08",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_median": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-20_20-17-00",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_trimmed_mean": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-20_20-17-58",
}

ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_trimmed_mean"
ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_trimmed_mean"


import re

exp_log = open(f"{ckpt_name_dir_map[ckpt_name]}/main_kma.log", "r").readlines()

cur_round = -1
match_result = {}
for line in exp_log:
    if "Round" in line:
        # match "Round 1 : [0, 2, 4, 6, 9]"
        round_str = re.search(r'Round (\d+)', line)

        pattern = r"Round\s+(?P<round_number>\d+)\s*:\s*(?P<list>\[[\d,\s]+\])"
        match = re.search(pattern, line)

        if match:
            round_number = match.group("round_number")
            list_string = match.group("list")
            list_data = ast.literal_eval(list_string)
            # print(f"Round Number: {round_number}, List: {list_string}")

            cur_round = round_number
            match_result[cur_round + "_participants"] = list_data

import json
import torch 
total_round = 20
max_compromised_clients = 2
number_of_clients = 5
threshold_factor = 0.5
device = "cpu"
metric_name = "total_acc"
deviation = 0.05

ckpt_dir = ckpt_name_dir_map[ckpt_name]
all_metrics_path = f"{ckpt_dir}/evaluation_false_acc_NoSysQA.json"
all_metrics = json.load(open(all_metrics_path, "r"))
# parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']
parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']

fact = list(all_metrics[0]["global"].keys())[0]


def get_params(keys, param_dict):
    return param_dict[keys[0]]

t = 0

client_results = {}

for round in range(1, total_round+1):
    participants_clients = match_result[f"{round}_participants"]


    attack_client_idxs = []
    for i in range(max_compromised_clients):
        if i in participants_clients:
            attack_client_idxs.append(i)

    match_result[f"{round}_attack_client_idxs"] = attack_client_idxs
    if len(attack_client_idxs) > 0:
        select_attack_idx = attack_client_idxs[0]
        
        t_nd = round - 1

        #上一次攻击时，初始全局模型
        theta_t = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[-2]

        #当前轮次初始全局模型
        theta_t_nd = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t_nd+1}.pth"), map_location=torch.device(device))[-2]

        #上一轮次恶意客户端模型
        theta_t_local = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[select_attack_idx]

        params_t = get_params(parameter_keys, theta_t)
        params_t_nd = get_params(parameter_keys, theta_t_nd)
        params_t_local = get_params(parameter_keys, theta_t_local)
        
        
        from utils.utils import read_indicator

        performance_feedback, param_feedback = read_indicator(t, t_nd, params_t, params_t_nd, params_t_local, number_of_clients=number_of_clients, threshold_factor=threshold_factor, all_metrics=all_metrics, fact=fact, metric_name=metric_name, deviation=deviation)

        t_participants = match_result[f"{t+1}_participants"]
        t_attack_client_idxs = match_result[f"{t+1}_attack_client_idxs"]

        print(f"Round {t+1} : participants: {t_participants}, attack_client_idxs: {t_attack_client_idxs}, performance_feedback: {performance_feedback}, param_feedback: {param_feedback}")

        t = t_nd
        # break


Round 1 : participants: [0, 2, 4, 6, 9], attack_client_idxs: [0], performance_feedback: False, param_feedback: False
Round 1 : participants: [0, 2, 4, 6, 9], attack_client_idxs: [0], performance_feedback: False, param_feedback: False
metric_t_nd: 0.0, metric_t: 0.0
p: 0.2959914207458496, threshold: 0.1
Round 2 : participants: [0, 1, 2, 3, 4], attack_client_idxs: [0, 1], performance_feedback: False, param_feedback: True
metric_t_nd: 0.02, metric_t: 0.0
p: 0.2638781666755676, threshold: 0.1
Round 3 : participants: [0, 1, 2, 7, 8], attack_client_idxs: [0, 1], performance_feedback: False, param_feedback: True
metric_t_nd: 0.0, metric_t: 0.02
p: 0.18494108319282532, threshold: 0.1
Round 5 : participants: [1, 3, 4, 5, 9], attack_client_idxs: [1], performance_feedback: False, param_feedback: True
metric_t_nd: 0.22, metric_t: 0.0
p: 0.042192064225673676, threshold: 0.1
Round 7 : participants: [1, 2, 6, 7, 9], attack_client_idxs: [1], performance_feedback: True, param_feedback: False
metric_t_n

## Median


In [6]:
import ast


ckpt_name_dir_map = {
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_multi-krum": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-20_23-57-25",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_multi-krum": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-29_02-15-17",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_median": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-31_00-46-34",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_trimmed_mean": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-31_00-48-08",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_median": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-20_20-17-00",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_trimmed_mean": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-20_20-17-58",
}

# ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_median"
ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_median"


import re

exp_log = open(f"{ckpt_name_dir_map[ckpt_name]}/main_kma.log", "r").readlines()

cur_round = -1
match_result = {}
for line in exp_log:
    if "Round" in line:
        # match "Round 1 : [0, 2, 4, 6, 9]"
        round_str = re.search(r'Round (\d+)', line)

        pattern = r"Round\s+(?P<round_number>\d+)\s*:\s*(?P<list>\[[\d,\s]+\])"
        match = re.search(pattern, line)

        if match:
            round_number = match.group("round_number")
            list_string = match.group("list")
            list_data = ast.literal_eval(list_string)
            # print(f"Round Number: {round_number}, List: {list_string}")

            cur_round = round_number
            match_result[cur_round + "_participants"] = list_data

import json
import torch 
total_round = 20
max_compromised_clients = 2
number_of_clients = 5
threshold_factor = 0.5
device = "cpu"
metric_name = "total_acc"
deviation = 0.05

ckpt_dir = ckpt_name_dir_map[ckpt_name]
all_metrics_path = f"{ckpt_dir}/evaluation_false_acc_NoSysQA.json"
all_metrics = json.load(open(all_metrics_path, "r"))
# parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']
parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']

fact = list(all_metrics[0]["global"].keys())[0]


def get_params(keys, param_dict):
    return param_dict[keys[0]]

t = 0

client_results = {}

for round in range(1, total_round+1):
    participants_clients = match_result[f"{round}_participants"]


    attack_client_idxs = []
    for i in range(max_compromised_clients):
        if i in participants_clients:
            attack_client_idxs.append(i)

    match_result[f"{round}_attack_client_idxs"] = attack_client_idxs
    if len(attack_client_idxs) > 0:
        select_attack_idx = attack_client_idxs[0]
        
        t_nd = round - 1

        #上一次攻击时，初始全局模型
        theta_t = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[-2]

        #当前轮次初始全局模型
        theta_t_nd = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t_nd+1}.pth"), map_location=torch.device(device))[-2]

        #上一轮次恶意客户端模型
        theta_t_local = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[select_attack_idx]

        params_t = get_params(parameter_keys, theta_t)
        params_t_nd = get_params(parameter_keys, theta_t_nd)
        params_t_local = get_params(parameter_keys, theta_t_local)
        
        
        from utils.utils import read_indicator

        performance_feedback, param_feedback = read_indicator(t, t_nd, params_t, params_t_nd, params_t_local, number_of_clients=number_of_clients, threshold_factor=threshold_factor, all_metrics=all_metrics, fact=fact, metric_name=metric_name, deviation=deviation)

        t_participants = match_result[f"{t+1}_participants"]
        t_attack_client_idxs = match_result[f"{t+1}_attack_client_idxs"]

        print(f"Round {t+1} : participants: {t_participants}, attack_client_idxs: {t_attack_client_idxs}, performance_feedback: {performance_feedback}, param_feedback: {param_feedback}")

        t = t_nd
        # break


Round 1 : participants: [0, 2, 4, 6, 9], attack_client_idxs: [0], performance_feedback: False, param_feedback: False
Round 1 : participants: [0, 2, 4, 6, 9], attack_client_idxs: [0], performance_feedback: False, param_feedback: False
metric_t_nd: 0.0, metric_t: 0.0
p: 0.15122394263744354, threshold: 0.1
Round 2 : participants: [0, 1, 2, 3, 4], attack_client_idxs: [0, 1], performance_feedback: False, param_feedback: True
metric_t_nd: 0.0, metric_t: 0.0
p: 0.1713356077671051, threshold: 0.1
Round 3 : participants: [0, 1, 2, 7, 8], attack_client_idxs: [0, 1], performance_feedback: False, param_feedback: True
metric_t_nd: 0.0, metric_t: 0.0
p: 0.07218633592128754, threshold: 0.1
Round 5 : participants: [1, 3, 4, 5, 9], attack_client_idxs: [1], performance_feedback: False, param_feedback: False
metric_t_nd: 0.0, metric_t: 0.0
p: 0.04468037188053131, threshold: 0.1
Round 7 : participants: [1, 2, 6, 7, 9], attack_client_idxs: [1], performance_feedback: False, param_feedback: False
metric_t_nd

## RFLBAT

In [7]:
import ast


ckpt_name_dir_map = {
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_rflbat": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-21_03-25-05",
    "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53_ele_norm_0.005.yaml_rflbat": "output/FinGPT/fingpt-sentiment-train_20000_fedavg_c10s5_i10_b4a4_l1024_r32a64_attack_edit_2025-03-31_02-29-47",
}

ckpt_name = "./attack/edit/hparams/FT-Plus/qwen2.5_3b_lora_20_rephrase_path_split53.yaml_rflbat"


import re

exp_log = open(f"{ckpt_name_dir_map[ckpt_name]}/main_kma.log", "r").readlines()

cur_round = -1
match_result = {}
for line in exp_log:
    if "Round" in line:
        # match "Round 1 : [0, 2, 4, 6, 9]"
        round_str = re.search(r'Round (\d+)', line)

        pattern = r"Round\s+(?P<round_number>\d+)\s*:\s*(?P<list>\[[\d,\s]+\])"
        match = re.search(pattern, line)

        if match:
            round_number = match.group("round_number")
            list_string = match.group("list")
            list_data = ast.literal_eval(list_string)
            # print(f"Round Number: {round_number}, List: {list_string}")

            cur_round = round_number
            match_result[cur_round + "_participants"] = list_data

import json
import torch 
total_round = 20
max_compromised_clients = 2
number_of_clients = 5
threshold_factor = 0.5
device = "cpu"
metric_name = "total_acc"
deviation = 0.05

ckpt_dir = ckpt_name_dir_map[ckpt_name]
all_metrics_path = f"{ckpt_dir}/evaluation_false_acc_NoSysQA.json"
all_metrics = json.load(open(all_metrics_path, "r"))
# parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']
parameter_keys = ['base_model.model.model.layers.27.mlp.down_proj.lora_B.weight']

fact = list(all_metrics[0]["global"].keys())[0]


def get_params(keys, param_dict):
    return param_dict[keys[0]]

t = 0

client_results = {}

for round in range(1, total_round+1):
    participants_clients = match_result[f"{round}_participants"]


    attack_client_idxs = []
    for i in range(max_compromised_clients):
        if i in participants_clients:
            attack_client_idxs.append(i)

    match_result[f"{round}_attack_client_idxs"] = attack_client_idxs
    if len(attack_client_idxs) > 0:
        select_attack_idx = attack_client_idxs[0]
        
        t_nd = round - 1

        #上一次攻击时，初始全局模型
        theta_t = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[-2]

        #当前轮次初始全局模型
        theta_t_nd = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t_nd+1}.pth"), map_location=torch.device(device))[-2]

        #上一轮次恶意客户端模型
        theta_t_local = torch.load(os.path.join(ckpt_dir, f"locals/local_dict_list_{t+1}.pth"), map_location=torch.device(device))[select_attack_idx]

        params_t = get_params(parameter_keys, theta_t)
        params_t_nd = get_params(parameter_keys, theta_t_nd)
        params_t_local = get_params(parameter_keys, theta_t_local)
        
        
        from utils.utils import read_indicator

        performance_feedback, param_feedback = read_indicator(t, t_nd, params_t, params_t_nd, params_t_local, number_of_clients=number_of_clients, threshold_factor=threshold_factor, all_metrics=all_metrics, fact=fact, metric_name=metric_name, deviation=deviation)

        t_participants = match_result[f"{t+1}_participants"]
        t_attack_client_idxs = match_result[f"{t+1}_attack_client_idxs"]

        print(f"Round {t+1} : participants: {t_participants}, attack_client_idxs: {t_attack_client_idxs}, performance_feedback: {performance_feedback}, param_feedback: {param_feedback}")

        t = t_nd
        # break

Round 1 : participants: [0, 2, 4, 6, 9], attack_client_idxs: [0], performance_feedback: False, param_feedback: False
Round 1 : participants: [0, 2, 4, 6, 9], attack_client_idxs: [0], performance_feedback: False, param_feedback: False
metric_t_nd: 0.0, metric_t: 0.0
p: 0.30215394496917725, threshold: 0.1
Round 2 : participants: [0, 1, 2, 3, 4], attack_client_idxs: [0, 1], performance_feedback: False, param_feedback: True
metric_t_nd: 0.28, metric_t: 0.0
p: 0.3327934145927429, threshold: 0.1
Round 3 : participants: [0, 1, 2, 7, 8], attack_client_idxs: [0, 1], performance_feedback: True, param_feedback: True
metric_t_nd: 0.52, metric_t: 0.28
p: 0.224792942404747, threshold: 0.1
Round 5 : participants: [1, 3, 4, 5, 9], attack_client_idxs: [1], performance_feedback: True, param_feedback: True
metric_t_nd: 0.94, metric_t: 0.52
p: 0.06388366222381592, threshold: 0.1
Round 7 : participants: [1, 2, 6, 7, 9], attack_client_idxs: [1], performance_feedback: True, param_feedback: False
metric_t_nd: